# Week 5 — Model Comparison

Compares three regressors on exactly the Part A data, split, and preprocessing (`outputs/02_preprocessed_v2.csv` + `model_pipeline.py`):

- **Validation phase:** train on 2025-04..2026-03 (train-only price filter), score on 2026-04. All hyperparameter and feature choices are made here.
- **Final phase:** refit on 2025-05..2026-04 (train-only price filter) and score the 2026-05 test month **once per final model**.
- All models predict `log(ClosePrice)` and are scored back in dollars, so the comparison with the Week 4 baseline is like-for-like.
- **Metrics used for decisions:** validation MAE, plus MdAPE as a secondary check. Validation R² is shown but is unreliable: three April 2026 sales of $645M–$796M make up about 98% of April's total variance, so R² stays near 0 for every model.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

import model_pipeline as mp

RANDOM_STATE = 42
MIN_FREQUENCY = 5
ORIGINAL_TEST_R2 = 0.2295
LR_FEATURES = mp.NUMERIC_FEATURES + mp.LOCATION_CATEGORICAL
TREE_FEATURES = mp.NUMERIC_FEATURES + mp.COORDINATE_FEATURES

data = pd.read_csv('outputs/02_preprocessed_v2.csv', dtype={'PostalCode': 'object'})
val_train, _ = mp.filter_train_prices(data.loc[data['val_split'].eq('train')])
val_rows = data.loc[data['val_split'].eq('val')]
train, train_bounds = mp.filter_train_prices(data.loc[data['split'].eq('train')])
test_rows = data.loc[data['split'].eq('test')]
print(f'val-train {len(val_train):,} | val {len(val_rows):,} | train {len(train):,} | test {len(test_rows):,}')

val-train 127,734 | val 12,021 | train 127,939 | test 12,015


In [2]:
def tree_model(estimator, use_zip=False):
    """Shared preprocessor (train-fit clip/impute/scale) + optional ZIP one-hot, log target."""
    prep = mp.make_preprocessor(numeric=mp.NUMERIC_FEATURES, log_features=[],
                                categorical=mp.LOCATION_CATEGORICAL if use_zip else [],
                                min_frequency=MIN_FREQUENCY, extra_numeric=mp.COORDINATE_FEATURES)
    return TransformedTargetRegressor(Pipeline([('prep', prep), ('model', estimator)]), func=np.log, inverse_func=np.exp)


def validate(model, cols):
    """Fit on validation-phase train rows, score on April 2026."""
    start = time.perf_counter()
    model.fit(val_train[cols], val_train[mp.TARGET])
    seconds = time.perf_counter() - start
    return {**mp.regression_metrics(val_rows[mp.TARGET], model.predict(val_rows[cols])), 'fit_s': seconds}


def final_fit_and_test(name, model, cols, val_metrics):
    """Refit on the final train window and score the test month once."""
    start = time.perf_counter()
    model.fit(train[cols], train[mp.TARGET])
    fit_s = time.perf_counter() - start
    train_m = mp.regression_metrics(train[mp.TARGET], model.predict(train[cols]))
    test_m = mp.regression_metrics(test_rows[mp.TARGET], model.predict(test_rows[cols]))
    row = {'model': name, 'train R2': train_m['R2'], 'val R2': val_metrics['R2'], 'val MAE': val_metrics['MAE'],
           'val MdAPE': val_metrics['MdAPE'], 'test R2': test_m['R2'], 'test MAE': test_m['MAE'],
           'test MdAPE': test_m['MdAPE'], 'final fit (s)': fit_s}
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    return row

results = []
fmt = {'R2': '{:.4f}', 'MAE': '${:,.0f}', 'MdAPE': '{:.2%}', 'fit_s': '{:.1f}'}

## 1. Linear Regression (final Week 4 baseline)
Same configuration as `03_baseline_model.ipynb`: 9 numeric features plus ZIP one-hot (min frequency 5), log target, no tuning.

In [3]:
lr_val = validate(mp.make_linear_model(True, log_features=[], min_frequency=MIN_FREQUENCY), LR_FEATURES)
results.append(final_fit_and_test('LinearRegression', mp.make_linear_model(True, log_features=[], min_frequency=MIN_FREQUENCY),
                                  LR_FEATURES, lr_val))

{'model': 'LinearRegression', 'train R2': 0.842, 'val R2': 0.0095, 'val MAE': 422894.6605, 'val MdAPE': 0.0977, 'test R2': 0.486, 'test MAE': 235941.1504, 'test MdAPE': 0.098, 'final fit (s)': 0.3978}


## 2. Decision Tree — small grid on validation
Grid: `max_depth` ∈ {6, 10, 14, 18} × `min_samples_leaf` ∈ {5, 20, 50, 100}, selected by validation MAE.
Then one check: does adding the ZIP one-hot to the chosen tree lower validation MAE? The answer decides ZIP use for both tree models.

In [4]:
dt_grid = []
for depth in (6, 10, 14, 18):
    for leaf in (5, 20, 50, 100):
        est = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf, random_state=RANDOM_STATE)
        dt_grid.append({'max_depth': depth, 'min_samples_leaf': leaf, **validate(tree_model(est), TREE_FEATURES)})
dt_grid = pd.DataFrame(dt_grid).sort_values('MAE')
best_dt = dt_grid.iloc[0]
DT_PARAMS = {'max_depth': int(best_dt['max_depth']), 'min_samples_leaf': int(best_dt['min_samples_leaf'])}
print('Best tree params:', DT_PARAMS)
dt_grid.head(6).style.format(fmt).hide(axis='index')

Best tree params: {'max_depth': 18, 'min_samples_leaf': 5}


max_depth,min_samples_leaf,R2,MAE,MdAPE,fit_s
18,5,0.0078,"$445,835",10.46%,0.5
18,20,0.0077,"$450,991",10.53%,0.5
14,5,0.0077,"$453,012",11.08%,0.5
14,20,0.0076,"$459,950",11.26%,0.5
18,50,0.0075,"$463,327",11.06%,0.5
14,50,0.0074,"$468,430",11.62%,0.4


In [5]:
dt_zip_val = validate(tree_model(DecisionTreeRegressor(**DT_PARAMS, random_state=RANDOM_STATE), use_zip=True),
                      TREE_FEATURES + mp.LOCATION_CATEGORICAL)
USE_ZIP_FOR_TREES = dt_zip_val['MAE'] < best_dt['MAE']
print(f"Val MAE without ZIP: ${best_dt['MAE']:,.0f} | with ZIP: ${dt_zip_val['MAE']:,.0f} -> USE_ZIP_FOR_TREES = {USE_ZIP_FOR_TREES}")
TREE_COLS = TREE_FEATURES + (mp.LOCATION_CATEGORICAL if USE_ZIP_FOR_TREES else [])

Val MAE without ZIP: $445,835 | with ZIP: $443,501 -> USE_ZIP_FOR_TREES = True


In [6]:
dt_val = dt_zip_val if USE_ZIP_FOR_TREES else best_dt.to_dict()
results.append(final_fit_and_test('DecisionTree', tree_model(DecisionTreeRegressor(**DT_PARAMS, random_state=RANDOM_STATE),
                                                             USE_ZIP_FOR_TREES), TREE_COLS, dt_val))

{'model': 'DecisionTree', 'train R2': 0.9206, 'val R2': 0.0079, 'val MAE': 443500.6618, 'val MdAPE': 0.1039, 'test R2': 0.4303, 'test MAE': 256542.611, 'test MdAPE': 0.1032, 'final fit (s)': 2.1623}


## 3. Random Forest — small grid on validation
`n_estimators=200`, `n_jobs=-1`, `random_state=42`, `max_depth=None`. Only `min_samples_leaf` ∈ {3, 5, 10} is tuned, by validation MAE. One fit on about 128k rows took about 9 s in a timing check, so this grid is cheap.

In [7]:
rf_grid = []
for leaf in (3, 5, 10):
    est = RandomForestRegressor(n_estimators=200, min_samples_leaf=leaf, n_jobs=-1, random_state=RANDOM_STATE)
    rf_grid.append({'min_samples_leaf': leaf, **validate(tree_model(est, USE_ZIP_FOR_TREES), TREE_COLS)})
rf_grid = pd.DataFrame(rf_grid).sort_values('MAE')
RF_LEAF = int(rf_grid.iloc[0]['min_samples_leaf'])
print('Best RF min_samples_leaf:', RF_LEAF)
rf_grid.style.format(fmt).hide(axis='index')

Best RF min_samples_leaf: 3


min_samples_leaf,R2,MAE,MdAPE,fit_s
3,0.0081,"$407,159",8.16%,88.8
5,0.0081,"$409,941",8.32%,58.6
10,0.0079,"$417,937",8.59%,44.0


In [8]:
rf_model = tree_model(RandomForestRegressor(n_estimators=200, min_samples_leaf=RF_LEAF, n_jobs=-1,
                                            random_state=RANDOM_STATE), USE_ZIP_FOR_TREES)
results.append(final_fit_and_test('RandomForest', rf_model, TREE_COLS, rf_grid.iloc[0].to_dict()))

pipe = rf_model.regressor_
importances = pd.Series(pipe.named_steps['model'].feature_importances_,
                        index=pipe.named_steps['prep'].get_feature_names_out()).sort_values(ascending=False)
importances.head(10).rename('importance').to_frame().style.format('{:.3f}')

{'model': 'RandomForest', 'train R2': 0.9513, 'val R2': 0.0081, 'val MAE': 407159.4218, 'val MdAPE': 0.0816, 'test R2': 0.4445, 'test MAE': 220620.919, 'test MdAPE': 0.0819, 'final fit (s)': 79.9609}


,importance
num__Latitude,0.314
num__LivingArea,0.285
num__Longitude,0.236
num__Bathrooms,0.080
num__LotSize,0.021
num__YearBuilt,0.014
num__PoolPrivateYN,0.007
num__Bedrooms,0.003
num__missingindicator_PoolPrivateYN,0.003
num__ParkingTotal,0.003


## 4. Results

In [9]:
table = pd.DataFrame(results)
reference = pd.DataFrame([{'model': 'v1 baseline (Week 4 original, reference)', 'test R2': ORIGINAL_TEST_R2}])
table = pd.concat([reference, table], ignore_index=True).set_index('model')
table = table[['train R2', 'val R2', 'val MAE', 'val MdAPE', 'test R2', 'test MAE', 'test MdAPE', 'final fit (s)']]
table.style.format({'train R2': '{:.4f}', 'val R2': '{:.4f}', 'val MAE': '${:,.0f}', 'val MdAPE': '{:.2%}',
                    'test R2': '{:.4f}', 'test MAE': '${:,.0f}', 'test MdAPE': '{:.2%}', 'final fit (s)': '{:.1f}'},
                   na_rep='n/a')

,train R2,val R2,val MAE,val MdAPE,test R2,test MAE,test MdAPE,final fit (s)
model,,,,,,,,
"v1 baseline (Week 4 original, reference)",n/a,n/a,n/a,n/a,0.2295,n/a,n/a,n/a
LinearRegression,0.8420,0.0095,"$422,895",9.77%,0.4860,"$235,941",9.80%,0.4
DecisionTree,0.9206,0.0079,"$443,501",10.39%,0.4303,"$256,543",10.32%,2.2
RandomForest,0.9513,0.0081,"$407,159",8.16%,0.4445,"$220,621",8.19%,80.0


## 5. Discussion

**Which metrics the conclusions rest on:** validation MAE (all choices) and test MAE and MdAPE (final comparison). Validation R² is about 0.01 for every model, because three April sales of \$645M–\$796M make up about 98% of April's total variance, so it can't separate the models. Train R² is measured on price-filtered training rows (\$235k–\$8.9M), so the train-to-test gaps below partly reflect the test month's wider price range.

### Linear Regression (Week 4 baseline)
- **Strengths:** highest test R² (**0.4860**). Because it is linear in log-price, it can predict above the training price range, which matters for the 44 test sales over \$10M (up to \$98M) that dominate R². Fastest by far (0.4 s). Fully interpretable: each coefficient is a percentage effect, including a price premium for every ZIP.
- **Weaknesses:** worse typical error than the RF (test MAE \$235,941 vs \$220,621; MdAPE 9.80% vs 8.19%). Effects are additive in log space, so it misses interactions such as size × location. Train R² 0.842 vs test 0.486.

### Decision Tree (max_depth=18, min_samples_leaf=5, + ZIP)
- **Choice:** best of the 16-cell grid by validation MAE (\$445,835 → \$443,501 with ZIP). The best cell is at the grid edge (deepest, smallest leaves), so a slightly deeper tree might help, but the grid was kept small on purpose.
- **Strengths:** fast (2.1 s). In principle the rules are human-readable.
- **Weaknesses:** worst on every test metric (R² **0.4303**, MAE \$256,543, MdAPE 10.32%). It has the largest drop from train to test R² (0.921 → 0.430), which points to high variance. At depth 18 the tree has thousands of leaves, so in practice it isn't interpretable. Like any tree on a log target, it can't predict above about \$8.9M (the training cap).

### Random Forest (200 trees, min_samples_leaf=3, + ZIP)
- **Strengths:** lowest typical error: validation MAE \$407,159, test MAE **\$220,621** (6.5% below LR), test MdAPE **8.19%** (vs 9.80%). Averaging 200 trees fixes most of the single tree's variance (test MAE \$36k lower than the DT).
- **Weaknesses:** test R² **0.4445** is below LR's. Its predictions are capped near the \$8.9M training maximum, so it badly underestimates the \$10M–\$98M sales that dominate R². It also fits the training data most closely (train R² 0.951) and is slowest (80.0 s in the saved run; 87–91 s in earlier runs). Adding the sparse ZIP one-hot raised fit time from about 9 s to about 87 s, for only about 0.5% lower validation MAE (\$409,061 without ZIP in the timing check vs \$407,159 with it). It is less interpretable and relies on feature importances.
- **Top-10 importances:** Latitude 0.314, LivingArea 0.285, Longitude 0.236, Bathrooms 0.080, LotSize 0.021, YearBuilt 0.014, PoolPrivateYN 0.007, Bedrooms 0.003, Pool-missing flag 0.003, ParkingTotal 0.003. Location comes almost entirely from latitude/longitude; no single ZIP dummy makes the top 10.

### Conclusion
For a typical home (MAE, MdAPE), **Random Forest is best**. For R² over all test sales, **Linear Regression is best**, because only it can extrapolate to the luxury tail. Every model clearly beats the original v1 baseline (test R² 0.2295). One option to explore next would be to train the trees without the upper price cap, or to blend LR and RF predictions for very high-value homes.